> **Note.** The random forest scores here are inflated by near-duplicate segments that the folds did not keep together (see notebook 06). The chance and logistic regression rows are not affected.

In [1]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from src.config import PROCESSED, RANDOM_STATE
from src.cv import load_fold_table
from src.data import feature_cols
from src.evaluate import cross_validate

dev = pd.read_csv(PROCESSED / "dev_2v3.csv")
cols = feature_cols(dev)
X = dev[cols].to_numpy(dtype=float)  
table = load_fold_table()   

In [2]:
models = {
    "chance (predicts the class prior)": lambda: DummyClassifier(strategy="prior"),
    "logistic regression, raw values": lambda: make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    "random forest (50 trees), raw values": lambda: RandomForestClassifier(50, random_state=RANDOM_STATE, n_jobs=-1),
}

results = {name: cross_validate(make_model, dev, X, table) for name, make_model in models.items()}

summary = pd.DataFrame({
    name: {"AUC": r["auc"].mean(), "AUC sd": r["auc"].std(), "accuracy": r["accuracy"].mean(),
           "chunk accuracy": r["chunk_accuracy"].mean(),
           "TP": r["tp"].mean(), "FN": r["fn"].mean(), "FP": r["fp"].mean(), "TN": r["tn"].mean()}
    for name, r in results.items()
}).T.round(3)
summary

,AUC,AUC sd,accuracy,chunk accuracy,TP,FN,FP,TN
chance (predicts the class prior),0.500,0.000,0.500,0.500,0.0,80.0,0.0,80.0
"logistic regression, raw values",0.516,0.016,0.495,0.502,34.2,45.8,35.0,45.0
"random forest (50 trees), raw values",0.778,0.021,0.690,0.608,50.2,29.8,19.8,60.2


In [3]:
seg_y = dev.groupby("segment")["y"].first()
shuffled_aucs = []
for s in range(4):
    rng = np.random.default_rng(100 + s)
    shuffled = pd.Series(rng.permutation(seg_y.values), index=seg_y.index)   # labels shuffled across segments
    d = dev.copy()
    d["y"] = d["segment"].map(shuffled)
    r = cross_validate(models["random forest (50 trees), raw values"], d, X, table, n_repeats=1)
    shuffled_aucs.append(r["auc"].iloc[0])
print(np.round(shuffled_aucs, 3), "mean", round(np.mean(shuffled_aucs), 3))   # should hover around 0.5

[0.477 0.422 0.541 0.458] mean 0.474


# 04 Baselines

Same setup for every model: development data only, 10 outer folds x 5 repeats, scored per segment (mean of its 23 chunk probabilities). Scores are averages over the 5 repeats.

| Model | AUC (sd) | Accuracy | Chunk accuracy |
|---|---|---|---|
| Chance | 0.500 (0.000) | 0.500 | 0.500 |
| Logistic regression, raw values | 0.516 (0.016) | 0.495 | 0.502 |
| Random forest (50 trees), raw values | 0.778 (0.021) | 0.690 | 0.608 |

Random forest confusion matrix (class 2 = positive, out of 160 segments): TP 50.2, FN 29.8, FP 19.8, TN 60.2.

## Findings
- Chance scores 0.5, as it should.
- Logistic regression on raw values is at chance. A linear model cannot use raw samples.
- The random forest is clearly above chance: AUC 0.78, and 69% of segments classified correctly. Averaging chunks into segments helps (69% per segment vs 61% per chunk).
- The forest finds class 3 segments more often than class 2 ones (75% vs 63%).
- Shuffled-label check: AUC 0.42 to 0.54 (mean 0.47). That is chance, so there is no sign of leakage.

## Caveats
- Untuned, 50 trees, development data only. The hold-out is untouched.
- With 160 segments, an AUC is uncertain by about +/- 0.07, more than the +/- 0.02 spread between repeats.
- It is not yet known what the forest relies on. The simple measures in the EDA (best AUC 0.63) do not explain it.